# Milestone 3 - Development Labels and Review Information

Current inputs are `data/labeled/suitability_review_master_v1.csv` for
review information and sampling metadata, and
`data/labeled/suitability_development_labels_v1.csv` for development labels.
Sampling, annotation suggestions and consolidation are already prepared;
older files are retained in `data/labeled/archive/`.
Do not rerun file-generation or annotation-generation code.

Manual verification was not performed for this experiment. The development
dataset combines three recorded first-review decisions with 46 automated
annotations; uncertain pairs are excluded. Evaluation will measure agreement
with these development labels, not independently verified suitability.
Reviewer assignments do not represent completed independent reviews.
This step only loads and validates existing inputs. No models are trained.


In [24]:
from pathlib import Path
import pandas as pd

# Find the project from the root or notebooks folder.
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data" / "labeled").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent

LABEL_DIR = PROJECT_ROOT / "data" / "labeled"
if not LABEL_DIR.is_dir():
    raise FileNotFoundError(f"Label folder not found: {LABEL_DIR}")


## Historical raw-data inspection — already completed

Optional provenance only; not required for the current workflow.
This saved code is non-executable and was not rerun.

```python
DATA_DIR = PROJECT_ROOT / "data" / "raw"

# Load only the columns needed for Milestone 3.
# Exclude the large precomputed embedding columns.

cv = pd.read_csv(
    DATA_DIR / "cv.csv",
    usecols=[
        "candidate_id",
        "resume_text",
        "clean_skills",
        "inferred_category",
    ],
)

jobs = pd.read_csv(
    DATA_DIR / "job.csv",
    usecols=[
        "job_id",
        "job_title",
        "job_text",
        "clean_skills",
        "category",
    ],
)

matches = pd.read_csv(
    DATA_DIR / "matches.csv",
    usecols=[
        "candidate_id",
        "job_id",
        "label_strength",
    ],
)

print("DATASET SIZES")
print(f"Resumes: {len(cv):,}")
print(f"Jobs: {len(jobs):,}")
print(f"Resume-job pairs: {len(matches):,}")

print("\nMISSING VALUES IN LOADED COLUMNS")
for name, frame in [
    ("Resumes", cv),
    ("Jobs", jobs),
    ("Matches", matches),
]:
    print(f"{name}: {int(frame.isna().sum().sum()):,}")

print("\nDUPLICATE IDENTIFIERS AND PAIRS")
print("Duplicate candidate IDs:", int(cv["candidate_id"].duplicated().sum()))
print("Duplicate job IDs:", int(jobs["job_id"].duplicated().sum()))
print(
    "Duplicate resume-job pairs:",
    int(matches.duplicated(["candidate_id", "job_id"]).sum()),
)

print("\nUNKNOWN REFERENCES")
print(
    "Pairs with unknown candidate IDs:",
    int((~matches["candidate_id"].isin(cv["candidate_id"])).sum()),
)
print(
    "Pairs with unknown job IDs:",
    int((~matches["job_id"].isin(jobs["job_id"])).sum()),
)

print("\nMATCH LABEL DISTRIBUTION")
label_counts = matches["label_strength"].value_counts(dropna=False)

label_summary = pd.DataFrame({
    "Count": label_counts,
    "Percentage": (label_counts / len(matches) * 100).round(2),
})

print(label_summary.to_string())

print("\nCANDIDATE CATEGORY DISTRIBUTION")
print(cv["inferred_category"].value_counts(dropna=False).to_string())

print("\nStep 2 completed. No files were changed.")
```

Historical saved output (not rerun):

```text
DATASET SIZES
Resumes: 9,544
Jobs: 1,167
Resume-job pairs: 122,560

MISSING VALUES IN LOADED COLUMNS
Resumes: 0
Jobs: 0
Matches: 0

DUPLICATE IDENTIFIERS AND PAIRS
Duplicate candidate IDs: 0
Duplicate job IDs: 0
Duplicate resume-job pairs: 0

UNKNOWN REFERENCES
Pairs with unknown candidate IDs: 0
Pairs with unknown job IDs: 0

MATCH LABEL DISTRIBUTION
                 Count  Percentage
label_strength                    
medium_positive  70636       57.63
weak_positive    51150       41.73
strong_positive    774        0.63

CANDIDATE CATEGORY DISTRIBUTION
inferred_category
INFORMATION-TECHNOLOGY    8140
FINANCE                   1010
SALES                      215
BUSINESS-DEVELOPMENT        83
HR                          80
UNKNOWN                     16

Step 2 completed. No files were changed.
```


In [25]:
# Load prepared inputs; preserve blanks and identifier strings.
review_path = LABEL_DIR / "suitability_review_master_v1.csv"
development_path = LABEL_DIR / "suitability_development_labels_v1.csv"

for file_path in [review_path, development_path]:
    if not file_path.is_file():
        raise FileNotFoundError(f"Required input not found: {file_path}")

review = pd.read_csv(review_path, keep_default_na=False, dtype=str)
training_data = pd.read_csv(
    development_path, keep_default_na=False, dtype=str
)

for name, frame in [("Master review", review), ("Development", training_data)]:
    if frame["pair_id"].duplicated().any():
        raise ValueError(f"Duplicate pair IDs in {name}.")
    expected_ids = frame["candidate_id"] + ":" + frame["job_id"]
    if not frame["pair_id"].eq(expected_ids).all():
        raise ValueError(f"Conflicting identifiers in {name}.")

if not training_data["pair_id"].isin(review["pair_id"]).all():
    raise ValueError("Development pairs are missing from the master.")
if not training_data["training_label"].isin(["0", "1"]).all():
    raise ValueError("Development labels must be binary; U is not 0.")
if not training_data["label_source"].isin([
    "existing_final_label", "recorded_first_review", "automated_annotation"
]).all():
    raise ValueError("Unknown development label source.")
for column in ["resume_text", "job_text"]:
    if training_data[column].str.strip().eq("").any():
        raise ValueError(f"Empty development document: {column}")

# Sampling metadata comes directly from the master.
sampling_columns = [
    "pair_id", "candidate_id", "job_id", "sampling_group",
    "sampling_domain", "sampling_reason", "candidate_inferred_domain",
    "source_match_recorded", "source_label_strength", "selection_version",
]
sampling = review[sampling_columns].copy()
if sampling["sampling_domain"].str.strip().eq("").any():
    raise ValueError("Missing sampling domain in the master.")

shared_columns = [
    "candidate_id", "job_id", "resume_text", "job_title",
    "job_text", "sampling_domain",
]
expected = review.set_index("pair_id").loc[
    training_data["pair_id"], shared_columns
]
actual = training_data.set_index("pair_id")[shared_columns]
if not actual.equals(expected):
    raise ValueError("Development documents or metadata differ from the master.")

reviewer_present = review["reviewer_1"].str.strip().ne("")
label_present = review["label_1"].str.strip().ne("")
untouched = ~reviewer_present & ~label_present
uncertain = reviewer_present & review["label_1"].str.strip().isin(["", "U"])

print("Master review pairs:", len(review))
print("Development pairs:", len(training_data))
print("Excluded from development:", len(review) - len(training_data))
print("Recorded first reviewers:", int(reviewer_present.sum()))
print("Uncertain first reviews:", int(uncertain.sum()))
print("Pairs without a first-review entry:", int(untouched.sum()))
print("Development label sources:")
print(training_data["label_source"].value_counts().to_string())
print("Automated annotations are not completed human reviews.")

# Summarize the prepared development labels.
label_counts = training_data["training_label"].value_counts()
print("\nDevelopment label counts:")
print("Suitable (1):", int(label_counts.get("1", 0)))
print("Unsuitable (0):", int(label_counts.get("0", 0)))

domain_counts = pd.crosstab(
    training_data["sampling_domain"], training_data["training_label"]
).reindex(columns=["1", "0"], fill_value=0)
domain_counts = domain_counts.rename(columns={"1": "Suitable", "0": "Unsuitable"})
domain_counts["Total"] = domain_counts.sum(axis=1)
print("\nDevelopment label counts by domain:")
print(domain_counts.to_string())


Master review pairs: 150
Development pairs: 49
Excluded from development: 101
Recorded first reviewers: 4
Uncertain first reviews: 1
Pairs without a first-review entry: 146
Development label sources:
label_source
automated_annotation     46
recorded_first_review     3
Automated annotations are not completed human reviews.

Development label counts:
Suitable (1): 12
Unsuitable (0): 37

Development label counts by domain:
training_label          Suitable  Unsuitable  Total
sampling_domain                                    
BUSINESS-DEVELOPMENT           1          13     14
FINANCE                        2           5      7
HR                             4           3      7
INFORMATION-TECHNOLOGY         5           9     14
SALES                          0           7      7


## Suitability Annotation Rubric

### Purpose

Assign a suitability label to each resume–job pair using evidence from
the resume and job description.

### Label Definitions

**1 — Suitable Match**

The resume demonstrates the core skills and relevant experience needed
for the role, with no clearly unmet essential requirement. Relevant
transferable skills may count when the reviewer explains the equivalence.
Missing optional or preferred skills alone do not make a pair unsuitable.

**0 — Unsuitable Match**

The resume shows a substantial mismatch with the role's core duties or
clearly does not meet an essential requirement. The reviewer must identify
the specific requirement and supporting evidence.

**Uncertain**

Leave the label blank when information is insufficient, contradictory,
or ambiguous. Record the reason and keep the pair pending. An omitted
skill phrase alone does not prove that the candidate lacks the skill.

### Review Rules

- Read the full resume and job description.
- Assess job-relevant skills, responsibilities, and experience.
- Distinguish essential requirements from optional preferences.
- Do not use personal or protected characteristics.
- Do not use source match strengths, similarity scores, or skill-coverage
  percentages as ground-truth suitability labels.
- Do not assume that an unrecorded pair is unsuitable.
- Record a short evidence-based rationale for each decision.

### Annotation method used in this experiment

The development dataset combines three recorded first-review
decisions with 46 automated annotations. Independent human
verification was not performed. Uncertain pairs are excluded.
Evaluation will measure agreement with these development labels;
it will not establish independently verified suitability.

### Interpretation

Labels describe suitability under this rubric. They do not represent
actual hiring decisions or probabilities of being hired.